# Boolean Feature Rule Miner — Full Demo
**EC2201 Unit I · Digital Fundamentals** (Boolean algebra, gates, truth tables, SOP/POS, K-map/Q-M simplification)

This notebook walks through the complete prototype, top to bottom:

1. **Digital-logic core** — AND/OR/NOT gates, expression parser & evaluator *(from scratch, no sklearn)*
2. **Gate-level evaluation trace** on one row
3. **Truth-table generation** (k ≤ 4 variables)
4. **SOP / POS conversion** (canonical forms)
5. **Boolean simplification** — identities + Quine–McCluskey
6. **Synthetic data** — hidden rule `Y = (A AND B) OR (C AND NOT D)` + 5% noise (seed 42)
7. **Rule mining** — brute-force candidates + min-support pruning, Support/Confidence/Coverage/F1
8. **Deep dive** on the best rule (truth table, SOP/POS, simplification, gate trace)
9. **Charts** — Top-10 bars, Coverage vs Confidence scatter, truth-table heatmap

> Run time: well under 2 minutes.

## 0 · Setup

In [1]:
import sys
from pathlib import Path

# locate the project root (works from any working directory)
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p / "boolean_logic.py").exists()), here)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import boolean_logic as bl
import rule_miner as rm
from data_generator import build_dataset
pd.set_option("display.width", 120)

Project root: /home/user/boolean-feature-rule-miner


## 1 · Digital-logic core — gates, parser, evaluator
The gates operate on **0/1 integers, NumPy arrays and Pandas Series**, so the
*same* gate used in a viva on one row also runs on the whole dataset.

**AND** is 1 only when both inputs are 1 · **OR** is 1 when at least one input is 1 · **NOT** inverts.

In [2]:
# --- gates on single bits ------------------------------------------------
print("Gates on single bits:")
print(f"  AND(1,1)={bl.AND(1,1)}  AND(1,0)={bl.AND(1,0)}  AND(0,0)={bl.AND(0,0)}")
print(f"  OR(0,1) ={bl.OR(0,1)}   OR(0,0) ={bl.OR(0,0)}   OR(1,1) ={bl.OR(1,1)}")
print(f"  NOT(1)  ={bl.NOT(1)}    NOT(0)  ={bl.NOT(0)}")

# --- the same gates on whole columns (element-wise) -----------------------
sample = pd.DataFrame({"A": [1, 0, 1, 1],
                       "B": [1, 1, 0, 0],
                       "C": [0, 1, 1, 1],
                       "D": [1, 0, 1, 0]})
sample["A AND B"]      = bl.AND(sample["A"], sample["B"])
sample["C AND NOT D"]  = bl.AND(sample["C"], bl.NOT(sample["D"]))
sample["rule"]         = bl.OR(sample["A AND B"], sample["C AND NOT D"])
print("\nGates on Pandas Series (element-wise over 4 rows):")
sample

Gates on single bits:
  AND(1,1)=1  AND(1,0)=0  AND(0,0)=0
  OR(0,1) =1   OR(0,0) =0   OR(1,1) =1
  NOT(1)  =0    NOT(0)  =1

Gates on Pandas Series (element-wise over 4 rows):


,A,B,C,D,A AND B,C AND NOT D,rule
0,1,1,0,1,1,0,1
1,0,1,1,0,0,1,1
2,1,0,1,1,0,0,0
3,1,0,1,0,0,1,1


## 2 · Expression evaluation + gate-level trace
`evaluate()` parses `(A AND B) OR (C AND NOT D)` (recursive-descent parser,
precedence NOT > AND > OR) and evaluates it.  `evaluate_row()` does the same
on **one row** and records every gate application — the digital-logic view.

In [3]:
expr = "(A AND B) OR (C AND NOT D)"
print("Rule:", expr)
print("Parsed variables:", bl.expression_variables(expr))

# evaluate on the sample frame (whole columns at once)
vals = bl.evaluate(expr, sample)
print("Vectorised evaluation on 4 rows:", list(vals))

# gate-level trace on ONE row: A=0, B=1, C=1, D=0  (rule fires)
row = {"A": 0, "B": 1, "C": 1, "D": 0}
out, trace = bl.evaluate_row(expr, row)
print(f"\nRow {row}  ->  F = {out}")
print(bl.format_trace(trace))

Rule: (A AND B) OR (C AND NOT D)
Parsed variables: ['A', 'B', 'C', 'D']
Vectorised evaluation on 4 rows: [1, 1, 0, 1]

Row {'A': 0, 'B': 1, 'C': 1, 'D': 0}  ->  F = 1
  Step 1 | Gate: AND | inputs: A=0, B=1                           | output: 0
  Step 2 | Gate: NOT | inputs: D=0                                | output: 1
  Step 3 | Gate: AND | inputs: C=1, NOT D=1                       | output: 1
  Step 4 | Gate: OR  | inputs: (A AND B)=0, (C AND NOT D)=1       | output: 1


## 3 · Truth-table generator (k ≤ 4 variables)
For a rule with k variables we enumerate all 2^k input combinations in binary
counting order and compute the output column `Y` with the gate evaluator.

In [4]:
tt = bl.generate_truth_table("A AND NOT B")
print("Truth table of (A AND NOT B):")
tt

tt_hidden = bl.generate_truth_table(expr)
print(f"\nTruth table of the hidden rule (k=4 -> {len(tt_hidden)} rows):")
tt_hidden

Truth table of (A AND NOT B):

Truth table of the hidden rule (k=4 -> 16 rows):


,A,B,C,D,Y
0,0,0,0,0,0
1,0,0,0,1,0
2,0,0,1,0,1
3,0,0,1,1,0
4,0,1,0,0,0
5,0,1,0,1,0
6,0,1,1,0,1
7,0,1,1,1,0
8,1,0,0,0,0
9,1,0,0,1,0


## 4 · SOP / POS conversion
Canonical forms are read straight off the truth table:
**SOP** = OR of the minterms where Y=1 · **POS** = AND of the maxterms where Y=0.

In [5]:
sop = bl.to_sop_canonical(expr)
pos = bl.to_pos_canonical(expr)
print("SOP:", sop["sum_notation"])
print("F =", sop["algebraic"])
print()
print("POS:", pos["product_notation"])
print("F =", pos["algebraic"])

SOP: Σm(2, 6, 10, 12, 13, 14, 15)
F = A'·B'·C·D' + A'·B·C·D' + A·B'·C·D' + A·B·C'·D' + A·B·C'·D + A·B·C·D' + A·B·C·D

POS: ΠM(0, 1, 3, 4, 5, 7, 8, 9, 11)
F = (A + B + C + D)·(A + B + C + D')·(A + B + C' + D')·(A + B' + C + D)·(A + B' + C + D')·(A + B' + C' + D')·(A' + B + C + D)·(A' + B + C + D')·(A' + B + C' + D')


## 5 · Boolean simplification (identities + Quine–McCluskey)
Classic textbook example: **(A AND B) OR (A AND NOT B) = A**
(factoring: A·(B + B') = A·1 = A).  The module does it *mechanically* with the
Quine–McCluskey method: minterms → combine patterns differing in one bit →
prime implicants → Petrick's method selects the minimal cover."


In [6]:
s = bl.simplify_expression("(A AND B) OR (A AND NOT B)")
print("BEFORE:", s["original"], f'({s["literal_before"]} literals)')
for line in s["steps"]:
    print("  ", line)
print("AFTER :", s["simplified_algebraic"], f'({s["literal_after"]} literals)')
print()
# identity-laws quick pass (educational)
si = bl.simplify_by_identities("(A AND B) OR (A AND NOT B)")
print("Identity pass:", si["original"], "->", si["simplified"])

BEFORE: (A AND B) OR (A AND NOT B) (4 literals)
   Minterms (F=1): 2(10), 3(11)
   Round 1: 10 + 11 -> 1-
   Round 2: no further combinations — remaining implicants are prime
AFTER : A (1 literals)

Identity pass: (A AND B) OR (A AND NOT B) -> (A AND B) OR (A AND NOT B)


## 6 · Synthetic data (seed 42, 500 rows, 5% label noise)
Ground truth: **Y = (A AND B) OR (C AND NOT D)**; E and F are pure noise
features that a good rule must ignore.  Reproducible via the seed.

In [7]:
df = build_dataset(n_rows=500, n_features=6, noise=0.05, seed=42)
print(f"rows={len(df)}  columns={list(df.columns)}  positive Y = {df['Y'].sum()} ({df['Y'].mean():.1%})")
df.head(10)

rows=500  columns=['A', 'B', 'C', 'D', 'E', 'F', 'Y']  positive Y = 226 (45.2%)


,A,B,C,D,E,F,Y
0,0,1,1,0,0,1,1
1,0,1,0,0,1,1,0
2,1,1,1,1,1,0,1
3,1,0,1,0,0,1,1
4,1,1,0,1,1,0,1
5,0,0,0,1,1,0,0
6,1,1,0,1,0,1,1
7,1,0,0,1,0,1,0
8,1,1,1,0,0,0,1
9,0,0,1,0,1,1,1


## 7 · Rule mining (brute force + min-support pruning)
Candidates: all conjunctions of 1–3 literals (232 here) **plus** OR-pairs of
two strong conjunctions (so multi-term SOP rules can be mined) **plus** TRUE.
Metrics per rule R → "Y=1":
Support = P(R & Y=1) · Confidence = P(Y=1|R) · Coverage = P(R) · Recall, F1.
Ranked by **F1 → Confidence → Support → fewer literals**.

In [8]:
res = rm.mine_rules(df, "Y", max_literals=3, min_support=0.05,
                   min_confidence=0.6, top_n=10, verbose=True)
print(f"\nRules surviving min_confidence=0.6: {len(res['rules'])}")
rm.format_rules_frame(res["top"])

  Target column : Y  (positive rows: 226/500 = 45.2%)


  Conjunction candidates : 232 generated, 232 passed min_support=0.05
  SOP (2-term) candidates : 190
  Constant candidates     : 1 (TRUE)
  Total candidates        : 423



Rules surviving min_confidence=0.6: 228


,#,Rule,n_literals,Support,Confidence,Coverage,Precision,Recall,F1
0,1,(A AND B) OR (C AND NOT D),4,0.432,0.9686,0.446,0.9686,0.9558,0.9621
1,2,(B) OR (C AND NOT D),3,0.440,0.7261,0.606,0.7261,0.9735,0.8318
2,3,(A AND B) OR (C),3,0.442,0.7199,0.614,0.7199,0.9779,0.8293
3,4,(A AND B) OR (NOT D),3,0.436,0.6877,0.634,0.6877,0.9646,0.8029
4,5,(B AND NOT E) OR (C AND NOT D),4,0.350,0.8294,0.422,0.8294,0.7743,0.8009
5,6,(B AND F) OR (C AND NOT D),4,0.356,0.7946,0.448,0.7946,0.7876,0.7911
6,7,(A AND B) OR (C AND F),4,0.350,0.7955,0.440,0.7955,0.7743,0.7848
7,8,(A AND B) OR (C AND NOT E),4,0.354,0.7832,0.452,0.7832,0.7832,0.7832
8,9,(A AND B) OR (C AND E),4,0.344,0.8037,0.428,0.8037,0.7611,0.7818
9,10,(A AND B) OR (NOT D AND NOT E),4,0.350,0.7883,0.444,0.7883,0.7743,0.7812


## 8 · Deep dive — the best rule
Truth table + SOP/POS + Q-M simplification + a gate-level trace on real data
rows (one where the rule fires, one where it does not)."


In [9]:
best = res["best"]
print("BEST RULE:", best["Rule"])
print(f"Support={best['Support']}  Confidence={best['Confidence']}  "
      f"Coverage={best['Coverage']}  Recall={best['Recall']}  F1={best['F1']}")

var_order = bl.expression_variables(best["Rule"])
tt = bl.generate_truth_table(best["Rule"], var_order)
print("\nTruth table:")
tt

sop = bl.to_sop_canonical(best["Rule"], var_order)
pos = bl.to_pos_canonical(best["Rule"], var_order)
print(f"\nSOP: {sop['sum_notation']}")
print(f"POS: {pos['product_notation']}")

s = bl.simplify_expression(best["Rule"], var_order)
print(f"\nSimplified: {s['simplified_algebraic']}  "
      f"({'already minimal' if s['already_minimal'] else 'reduced'})")
for line in s["steps"]:
    print("  ", line)

# gate trace on two real rows (one F=1, one F=0)
series = best["series"]
for idx in [series[series == 1].index[0], series[series == 0].index[0]]:
    row = {f: int(df.loc[idx, f]) for f in df.columns if f != "Y"}
    out, trace = bl.evaluate_row(best["Rule"], row)
    print(f"\nRow {idx}: {row} -> F={out}")
    print(bl.format_trace(trace))

BEST RULE: (A AND B) OR (C AND NOT D)
Support=0.432  Confidence=0.9686  Coverage=0.446  Recall=0.9558  F1=0.9621

Truth table:

SOP: Σm(2, 6, 10, 12, 13, 14, 15)
POS: ΠM(0, 1, 3, 4, 5, 7, 8, 9, 11)

Simplified: C·D' + A·B  (already minimal)
   Minterms (F=1): 2(0010), 6(0110), 10(1010), 12(1100), 13(1101), 14(1110), 15(1111)
   Round 1: 0010 + 0110 -> 0-10;  0010 + 1010 -> -010;  0110 + 1110 -> -110;  1010 + 1110 -> 1-10;  1100 + 1101 -> 110-;  1100 + 1110 -> 11-0;  1101 + 1111 -> 11-1;  1110 + 1111 -> 111-
   Round 2: 0-10 + 1-10 -> --10;  -010 + -110 -> --10;  110- + 111- -> 11--;  11-0 + 11-1 -> 11--
   Round 3: no further combinations — remaining implicants are prime

Row 0: {'A': 0, 'B': 1, 'C': 1, 'D': 0, 'E': 0, 'F': 1} -> F=1
  Step 1 | Gate: AND | inputs: A=0, B=1                           | output: 0
  Step 2 | Gate: NOT | inputs: D=0                                | output: 1
  Step 3 | Gate: AND | inputs: C=1, NOT D=1                       | output: 1
  Step 4 | Gate: OR  |

## 9 · Charts

In [10]:
import main as m   # chart helpers (also used by make_screenshots.py / app.py)

fig1 = m.plot_top_rules_bar(res["top"])
fig2 = m.plot_coverage_scatter(res["top"])
fig3 = m.plot_truth_table_heatmap(best["Rule"], var_order=var_order)
plt.show()

## 10 · Conclusion / acceptance checklist
- [x] Core digital-logic module written from scratch (gates, parser, evaluator)
- [x] Hidden rule `(A AND B) OR (C AND NOT D)` recovered at **rank #1** despite 5% label noise
- [x] Best rule shown with full truth table + SOP/POS + Q-M simplification
- [x] Gate-level trace on real data rows
- [x] 15/15 tests pass — `python tests/test_cases.py`
- [x] Charts in `screenshots/` — `python make_screenshots.py`

**Viva talking points:** the miner is an exhaustive (pruned) search over the
Boolean hypothesis space — the same machinery as K-map grouping, generalized
to arbitrary dataset sizes; Confidence of the recovered rule ≈ 0.97 ≈ the
(1 − noise) ceiling, which is exactly what a *perfect* rule should achieve
on 5%-noisy data.